In [ ]:
!pip install pyspark pyiceberg duckdb splink


In [ ]:
import os
from pyspark.sql import SparkSession

# Set up local Iceberg catalog simulation in PySpark
warehouse_path = "/tmp/ehdip_warehouse"
os.makedirs(warehouse_path, exist_ok=True)

spark = SparkSession.builder \
    .appName("Day01_Iceberg_Setup") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", warehouse_path) \
    .getOrCreate()

# Create Bronze Namespace
spark.sql("CREATE NAMESPACE IF NOT EXISTS local.ehdip_bronze")

# Create Bronze Raw Payload Table with audit metadata
spark.sql("""
CREATE TABLE IF NOT EXISTS local.ehdip_bronze.raw_payloads (
    payload_id STRING,
    source_system_id STRING,
    raw_payload_json STRING,
    ingestion_timestamp TIMESTAMP
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
""")

# Insert a mock record to verify
spark.sql("""
INSERT INTO local.ehdip_bronze.raw_payloads 
VALUES ('msg-001', 'EHR-A', '{"resourceType": "Patient", "id": "123"}', current_timestamp())
""")

df = spark.sql("SELECT * FROM local.ehdip_bronze.raw_payloads")
df.show(truncate=False)
